In [ ]:
%load_ext autoreload
%autoreload 2

## Training and Validation for Torchvision Maskrcnn on Volpy Data 
Tutorial for training and valdiation Maskrcnn model

@authors: Changjia Cai, Erik Thompson, and Manuel Paez

Date Created: August 28th, 2024

Date Updated: July 7th, 2025

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torchvision
from torch.optim.lr_scheduler import CyclicLR
from torch.utils.data import DataLoader

from config import Config
from model import get_model_instance_segmentation
from neurons import (
    NeuronsDataset,
    _atomic_torch_save,
    _check_output_directory,
    perform_final_evaluation,
    train_one_epoch,
    validate,
)
from utils import collate_fn, data_transform

#### Check if cuda is available

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

## Volpy Data, Training, and Validation Sets
There are 24 datasets in total, 3 types of different voltage imaging datasets recorded from mouse L1 cortex (L1), mouse hippocampus (HPC), and zebrafish tegmental area (TEG). Set up the image directories from https://zenodo.org/records/4515768 as follows:

    volpy_training_data/
        images/
            HPC.29.04.npz
            ...
        masks/
            HPC.29.04_mask.npz
            ...      

### Config 
The base configuration class `Config` from `config.py` contains the variables necessary for training and validation on the VolPy dataset. By default it reads training data from `caiman_datadir()/volpy_training_data` and writes checkpoints to `caiman_datadir()/model`. CaImAn's data directory can be changed with `CAIMAN_DATA`; the two VolPy locations can also be overridden independently with `CAIMAN_VOLPY_TRAINING_DATA` and `CAIMAN_VOLPY_MODEL_DIR`.

For data or model directories outside the standard CaImAn data directory, set portable environment variables before starting Jupyter (replace the example placeholders with paths appropriate for your system):

    export CAIMAN_DATA=/path/to/caiman_data
    export CAIMAN_VOLPY_TRAINING_DATA=/path/to/volpy_training_data
    export CAIMAN_VOLPY_MODEL_DIR=/path/to/new/model_output_directory

Training saves `mrcnn_latest.pt` and `volpy_train_history.pt` after every completed epoch, periodic `mrcnn_epoch_N.pt` snapshots according to `SAVE_FREQ`, and an epoch-numbered checkpoint for the final epoch. Existing training artifacts are protected by default; select a new `CAIMAN_VOLPY_MODEL_DIR` for a new run, or set `config.ALLOW_OVERWRITE = True` deliberately.

    Config: 
        # Paths
        DATA_DIR = caiman_datadir()/volpy_training_data
        MODEL_SAVE_DIR = caiman_datadir()/model

        # Model and Training Hyperparameters
        NUM_CLASSES = 1 + 1  # Background + Neuron
        BATCH_SIZE = 2 
        NUM_EPOCHS = 100
        MAX_LR = 0.005
        BASE_LR = 0.000001
        STEP_SIZE_UP = 3
        STEP_SIZE_DOWN = 7

        # Data Loading, Splitting, and Inference
        # IMAGES_PER_GPU = 2
        RANDOM_SPLIT = False # True for random split, False for fixed split from map below.
        NUM_TEST_RANDOM = 8
        NUM_TORCH_WORKERS = 4
        RANDOM_SEED = 42
        DATASET_REGION_MAP = {
            'HPC': [0, 1, 2, 3],
            'L1': [12, 13, 14],
            'TEG': [21],
            'Train': [4, 5, 6, 7, 8, 9, 10, 11, 15, 16, 17, 18, 19, 20, 22, 23]
        }

        INFERENCE_THRESHOLD = 0.5

        # Logging and Saving Frequency
        PRINT_FREQ = 1 
        SAVE_FREQ = 5

### Dataset 

Building on the standard `torch.utils.data.Dataset`class, we use 'NeuronsDataset' from neurons.py. The  `__getitem__` method of this class should return an `image` and a `target` dictionary delineating the different objects (box/mask) in the image:

    image: torchvision.tv_tensors.Image of shape [3, H, W]: can be a pure tensor, or a PIL Image of size (H, W)
    target: a dict containing the following keys
        masks : torchvision uint8 binary masks for each object (N,H,W) (N masks)
        boxes (bounding boxes)  (nx4)
        labels (int) label for each bounding box (note 0 is background, so if you have no bg, start with 1)
        image_id (int) unique image id
        area (float) area of bounding box 
        iscrowd (uint8) instances with `iscrowd=True` will be ignored during evaluation 

The 'data_transform' function from neurons.py should return a training pipeline for training. 
For windows, set the number of workers to 0. Otherwise, set to 4+

#### Dataset Split
The dataset split can be randomized split (such that each of L1, HPC, and TEG has a validation and training set) or can be set fixed. 

#### Data Transforms
From 'utils.py':

        transforms.append(T.RandomHorizontalFlip(p=0.5))
        transforms.append(T.RandomVerticalFlip(p=0.5))
        transforms.append(T.RandomApply([T.RandomRotation(degrees=(-5, 5))], p=0.5))
        transforms.append(T.ColorJitter(brightness=0.5,
                                        contrast=0.5,
                                        saturation=0.5,
                                        hue=0))
        transforms.append(T.GaussianBlur(kernel_size=(5, 5), sigma=(0.001, 0.3)))
        transforms.append(T.SanitizeBoundingBoxes(min_size=2))

#### Additional Training Setup:
Validation indices: [0, 1, 2, 3, 12, 13, 14, 21]

Training indices: [4, 5, 6, 7, 8, 9, 10, 11, 15, 16, 17, 18, 19, 20, 22, 23]

In [ ]:
config = Config()
if config.NUM_EPOCHS < 1:
    raise ValueError('NUM_EPOCHS must be at least 1')
if config.SAVE_FREQ < 1:
    raise ValueError('SAVE_FREQ must be at least 1')
_check_output_directory(
    config.MODEL_SAVE_DIR,
    config.ALLOW_OVERWRITE,
    num_epochs=config.NUM_EPOCHS,
    save_freq=config.SAVE_FREQ,
)
print(f'Saving training artifacts to: {os.path.abspath(config.MODEL_SAVE_DIR)}')
config.display()

np.random.seed(config.RANDOM_SEED)
torch.manual_seed(config.RANDOM_SEED)

print('Loading datasets...')
dataset_train = NeuronsDataset(config.DATA_DIR, data_transform(train=True))
dataset_val = NeuronsDataset(config.DATA_DIR, data_transform(train=False))

if config.RANDOM_SPLIT:
    print('Using random train/validation split.')
    indices = np.random.default_rng(config.RANDOM_SEED).permutation(len(dataset_train)).tolist()
    train_indices = indices[:-config.NUM_TEST_RANDOM]
    val_indices = indices[-config.NUM_TEST_RANDOM:]
else:
    print('Using fixed split from DATASET_REGION_MAP.')
    train_indices = config.DATASET_REGION_MAP['Train']
    val_indices = [
        idx
        for region, region_indices in config.DATASET_REGION_MAP.items()
        if region != 'Train'
        for idx in region_indices
    ]

val_indices_path = os.path.join(config.MODEL_SAVE_DIR, 'validation_indices.npy')
temporary_indices_path = f'{val_indices_path}.tmp.npy'
np.save(temporary_indices_path, val_indices)
os.replace(temporary_indices_path, val_indices_path)
print(f'Validation indices saved to {val_indices_path}')

train_dataset = torch.utils.data.Subset(dataset_train, train_indices)
val_dataset = torch.utils.data.Subset(dataset_val, val_indices)

shuffle_generator = torch.Generator().manual_seed(config.RANDOM_SEED)
data_loader_train = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    generator=shuffle_generator,
    num_workers=config.NUM_TORCH_WORKERS,
    collate_fn=collate_fn,
)
data_loader_val = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=config.NUM_TORCH_WORKERS,
    collate_fn=collate_fn,
)

### Model
We start from torchvision's COCO-pretrained Mask R-CNN FPN-v2 model, replace its box and mask predictors for background-plus-neuron classification, and fine-tune the trainable layers. 

### Optimizer + lr scheduler
We use the SGD optimizer and a CyclicLR scheduler.

In [ ]:
# Start from torchvision's COCO-pretrained Mask R-CNN and replace its heads
# with the two-class (background + neuron) VolPy heads.
model = get_model_instance_segmentation(config.NUM_CLASSES)
model.to(device)

params = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=config.MAX_LR,
    momentum=0.9,
    weight_decay=0.0001,
)
lr_scheduler = CyclicLR(
    optimizer,
    base_lr=config.BASE_LR,
    max_lr=config.MAX_LR,
    step_size_up=config.STEP_SIZE_UP,
    step_size_down=config.STEP_SIZE_DOWN,
    mode='triangular2',
)

### Training the Network

In [ ]:
all_train_losses, all_val_losses, all_lrs = [], [], []
history_path = os.path.join(config.MODEL_SAVE_DIR, 'volpy_train_history.pt')
latest_path = os.path.join(config.MODEL_SAVE_DIR, 'mrcnn_latest.pt')
final_model_path = os.path.join(
    config.MODEL_SAVE_DIR, f'mrcnn_epoch_{config.NUM_EPOCHS}.pt'
)
print(
    f'**TRAIN {config.NUM_EPOCHS} epochs. '
    f'PRINT every {config.PRINT_FREQ} epoch(s). '
    f'SAVE every {config.SAVE_FREQ} epoch(s).**'
)

for epoch in range(config.NUM_EPOCHS):
    train_loss = train_one_epoch(model, optimizer, data_loader_train, device, epoch)
    val_loss = validate(model, data_loader_val, device, epoch)
    current_lr = optimizer.param_groups[0]['lr']

    all_train_losses.append(train_loss)
    all_val_losses.append(val_loss)
    all_lrs.append(current_lr)
    lr_scheduler.step()

    completed_epoch = epoch + 1
    history = {
        'train_loss': all_train_losses,
        'val_loss': all_val_losses,
        'lr': all_lrs,
    }
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': lr_scheduler.state_dict(),
        'epoch': completed_epoch,
        'config': config.to_dict(),
        'torch_version': str(torch.__version__),
        'torchvision_version': str(torchvision.__version__),
        'train_indices': train_indices,
        'validation_indices': val_indices,
        'history': history,
    }

    # These writes are atomic: an interrupted write cannot replace a good checkpoint.
    _atomic_torch_save(checkpoint, latest_path)
    _atomic_torch_save(history, history_path)

    if completed_epoch % config.PRINT_FREQ == 0:
        print(
            f'Epoch {completed_epoch}/{config.NUM_EPOCHS} | '
            f'Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, '
            f'LR: {current_lr:.6f}'
        )

    if completed_epoch % config.SAVE_FREQ == 0 or completed_epoch == config.NUM_EPOCHS:
        model_path = os.path.join(
            config.MODEL_SAVE_DIR, f'mrcnn_epoch_{completed_epoch}.pt'
        )
        _atomic_torch_save(checkpoint, model_path)
        print(f'Checkpoint saved to {model_path}')

print(f'DONE! Final checkpoint: {final_model_path}')

### Plot Loss Function 

In [ ]:
plt.plot(np.array(all_train_losses), color='blue', marker='.', label='train')
plt.plot(np.array(all_val_losses), color='red', marker='.', label='validation')
plt.legend()
plt.xlabel('epoch')
plt.ylabel('net loss')
plt.title('loss function across different epochs')
plt.grid()

### Plot Learning Rate vs Epoch

In [ ]:
plt.plot(all_lrs, marker='.')
plt.xlabel('epoch')
plt.ylabel('learning rate')
plt.title('learning rate across different epochs')

### Inference using Test Set

For this, we will compute F1 scores for all datasets

#### Perform_final_evaluation 
From 'neurons.py', runs inference on the validtion set, calculates F1 scores for each region (i.e. HPC, L1, TEG), and reports the results. 

Note: Aim for 74 % >

In [ ]:
val_indices_path = os.path.join(config.MODEL_SAVE_DIR, 'validation_indices.npy')
if not os.path.exists(val_indices_path):
    raise FileNotFoundError(f'Validation indices not found at {val_indices_path}')

saved_val_indices = np.load(val_indices_path)
print(f'Evaluating {len(saved_val_indices)} validation images')
perform_final_evaluation(model, config, device, plot_results=True)